# Collect & Rerun Failed SWE-Bench Instances

Interactive notebook to:
1. Load `output.jsonl` from a rollout
2. Classify every instance by error type
3. Browse instances per error category
4. Selectively remove bad instances so the next rollout picks them up

## 1. Import Libraries & Configuration

In [1]:
import json
import os
import shutil
from datetime import datetime
from collections import Counter

import pandas as pd
from IPython.display import display, Markdown

# ====== CONFIGURATION — edit this path ======
OUTPUT_FILE = "/home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/GLM-5-FP8_maxiter_100_N_v0.61.0-no-hint-train-glm5_fp8_t05-run_1/output.jsonl"

# Derived paths
EVAL_FILE = OUTPUT_FILE.replace(".jsonl", ".swebench_eval.jsonl")
print(f"Output file: {OUTPUT_FILE}")
print(f"Eval file:   {EVAL_FILE}")
print(f"Output exists: {os.path.exists(OUTPUT_FILE)}")
print(f"Eval exists:   {os.path.exists(EVAL_FILE)}")

Output file: /home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/GLM-5-FP8_maxiter_100_N_v0.61.0-no-hint-train-glm5_fp8_t05-run_1/output.jsonl
Eval file:   /home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/GLM-5-FP8_maxiter_100_N_v0.61.0-no-hint-train-glm5_fp8_t05-run_1/output.swebench_eval.jsonl
Output exists: True
Eval exists:   True


## 2. Load Data & Classify Instances

In [2]:
def load_output_file(filepath: str) -> list[tuple[str, dict | None]]:
    """Load output.jsonl returning (raw_line, parsed_data_or_None) tuples."""
    entries = []
    with open(filepath, "r") as f:
        for line in f:
            raw = line.rstrip("\n")
            if not raw.strip():
                entries.append((raw, None))
                continue
            try:
                data = json.loads(raw)
                entries.append((raw, data))
            except json.JSONDecodeError:
                entries.append((raw, None))
    return entries


def classify_instance(data: dict) -> dict:
    """Classify an instance into error categories."""
    tr = data.get("test_result", {})
    patch = tr.get("git_patch", "")
    error = data.get("error")

    result = {
        "instance_id": data.get("instance_id", "unknown"),
        "empty_patch": not patch or not patch.strip(),
        "has_error": bool(error),
        "error_category": "no_error",
        "error_summary": "",
    }

    if error:
        err_str = str(error)
        if "Agent reached maximum iteration" in err_str:
            result["error_category"] = "max_iteration"
        elif "AgentStuckInLoopError" in err_str:
            result["error_category"] = "stuck_in_loop"
        elif "Maximum retries" in err_str and (
            "docker" in err_str.lower()
            or "buildx" in err_str.lower()
            or "Failed to cd" in err_str
            or "RuntimeError" in err_str
        ):
            result["error_category"] = "max_retries_infrastructure"
        elif "Maximum retries" in err_str and (
            "Retryable controller error" in err_str
            or "STATUS$ERROR_LLM" in err_str
        ):
            result["error_category"] = "max_retries_retryable"
        elif "Maximum retries" in err_str:
            result["error_category"] = "max_retries_other"
        elif "EvalTimeoutException" in err_str or "timed out" in err_str.lower():
            result["error_category"] = "timeout"
        else:
            result["error_category"] = "other_error"
        result["error_summary"] = err_str[:200]

    return result


# Load and classify
raw_entries = load_output_file(OUTPUT_FILE)
records = []
for raw, data in raw_entries:
    if data is not None:
        records.append(classify_instance(data))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} instances from output.jsonl")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 2119 instances from output.jsonl
Columns: ['instance_id', 'empty_patch', 'has_error', 'error_category', 'error_summary']


,instance_id,empty_patch,has_error,error_category,error_summary
0,getmoto__moto-5085,False,False,no_error,
1,getmoto__moto-5752,False,False,no_error,
2,getmoto__moto-4950,False,False,no_error,
3,getmoto__moto-6709,True,False,no_error,
4,getmoto__moto-7082,False,False,no_error,


## 3. Error Category Overview

Summary counts for each error category, cross-tabulated with empty/non-empty patch.

In [3]:
# Overall counts
print(f"Total instances: {len(df)}")
print(f"Empty patches:   {df['empty_patch'].sum()}")
print(f"Has error:       {df['has_error'].sum()}")
print()

# Cross-tab: error_category vs empty_patch
ct = pd.crosstab(df["error_category"], df["empty_patch"], margins=True, margins_name="Total")
ct.columns = ["has_patch", "empty_patch", "Total"]
display(Markdown("### Error Category × Empty Patch"))
display(ct)

# Simple category counts
print("\n### Error category counts:")
cat_counts = df["error_category"].value_counts()
for cat, cnt in cat_counts.items():
    print(f"  {cat}: {cnt}")

Total instances: 2119
Empty patches:   154
Has error:       715



### Error Category × Empty Patch

,has_patch,empty_patch,Total
error_category,,,
max_iteration,561,66,627
max_retries_infrastructure,0,42,42
max_retries_retryable,0,31,31
no_error,1399,5,1404
stuck_in_loop,5,10,15
Total,1965,154,2119



### Error category counts:
  no_error: 1404
  max_iteration: 627
  max_retries_infrastructure: 42
  max_retries_retryable: 31
  stuck_in_loop: 15


## 4. List Instances per Error Type

Browse the instance IDs and error summaries for each category. Modify the `category` variable to explore different types.

In [4]:
# Show all instances grouped by error category
for cat in df["error_category"].unique():
    subset = df[df["error_category"] == cat]
    empty_count = subset["empty_patch"].sum()
    patch_count = len(subset) - empty_count
    print(f"\n{'='*70}")
    print(f"Category: {cat}  (total: {len(subset)}, empty_patch: {empty_count}, has_patch: {patch_count})")
    print(f"{'='*70}")
    for _, row in subset.iterrows():
        tag = "[EMPTY]" if row["empty_patch"] else "[PATCH]"
        err = f" | {row['error_summary']}" if row["error_summary"] else ""
        print(f"  {tag} {row['instance_id']}{err}")


Category: no_error  (total: 1404, empty_patch: 5, has_patch: 1399)
  [PATCH] getmoto__moto-5085
  [PATCH] getmoto__moto-5752
  [PATCH] getmoto__moto-4950
  [EMPTY] getmoto__moto-6709
  [PATCH] getmoto__moto-7082
  [PATCH] getmoto__moto-6868
  [PATCH] getmoto__moto-6178
  [PATCH] getmoto__moto-6920
  [PATCH] getmoto__moto-5212
  [PATCH] getmoto__moto-5386
  [PATCH] getmoto__moto-7456
  [PATCH] getmoto__moto-5725
  [PATCH] getmoto__moto-6075
  [PATCH] getmoto__moto-6091
  [PATCH] getmoto__moto-5154
  [PATCH] getmoto__moto-6535
  [PATCH] getmoto__moto-5513
  [PATCH] getmoto__moto-5949
  [PATCH] getmoto__moto-7514
  [PATCH] getmoto__moto-4986
  [PATCH] getmoto__moto-5844
  [PATCH] getmoto__moto-6641
  [PATCH] getmoto__moto-7153
  [PATCH] getmoto__moto-7023
  [PATCH] getmoto__moto-7029
  [PATCH] getmoto__moto-6636
  [PATCH] getmoto__moto-4915
  [PATCH] getmoto__moto-6376
  [PATCH] getmoto__moto-5175
  [PATCH] getmoto__moto-5865
  [PATCH] getmoto__moto-5812
  [PATCH] getmoto__moto-5968
  [P

In [5]:
# Drill into a specific category — change this to explore
category = "no_error"  # <-- change me: no_error, max_iteration, stuck_in_loop, max_retries_infrastructure, max_retries_retryable, max_retries_other, timeout, other_error

subset = df[df["error_category"] == category].copy()
print(f"Category '{category}': {len(subset)} instances  (empty_patch: {subset['empty_patch'].sum()})")
display(subset[["instance_id", "empty_patch", "error_summary"]].reset_index(drop=True))

Category 'no_error': 1404 instances  (empty_patch: 5)


,instance_id,empty_patch,error_summary
0,getmoto__moto-5085,False,
1,getmoto__moto-5752,False,
2,getmoto__moto-4950,False,
3,getmoto__moto-6709,True,
4,getmoto__moto-7082,False,
...,...,...,...
1399,modin-project__modin-5952,False,
1400,modin-project__modin-6298,False,
1401,modin-project__modin-6764,False,
1402,modin-project__modin-6760,False,


## 5. Select Categories to Remove & Rerun

Choose which error categories to remove from `output.jsonl` (and its `.swebench_eval.jsonl`).

**Default policy:**
- **Always remove:** empty patches that are NOT from max_iteration/stuck_in_loop (agent didn't even try)
- **Optionally remove:** infrastructure errors, retryable LLM errors, timeouts
- **Never remove by default:** max_iteration, stuck_in_loop (agent ran but failed legitimately)

Edit `CATEGORIES_TO_REMOVE` and `ALSO_REMOVE_EMPTY_PATCH_WITH` below to control what gets removed.

In [ ]:
# ====== CONFIGURE WHAT TO REMOVE ======

# Remove all instances with these error categories (regardless of patch status)
CATEGORIES_TO_REMOVE = {
    # "max_retries_infrastructure",
    "max_retries_retryable",
    "max_retries_other",
    "timeout",
    # "other_error",        # uncomment to also remove other errors
    # "max_iteration",      # uncomment to also remove max_iteration (agent ran but failed)
    # "stuck_in_loop",      # uncomment to also remove stuck_in_loop
}

# Additionally remove all empty-patch instances EXCEPT those with these error categories
# (i.e., empty patches from max_iteration/stuck_in_loop are kept by default since agent ran)
KEEP_EMPTY_PATCH_IF_CATEGORY = {
    "max_iteration",
    "stuck_in_loop",
}

# ====== BUILD REMOVAL SET ======
rerun_ids = set()
removal_reasons = {}

for _, row in df.iterrows():
    iid = row["instance_id"]
    cat = row["error_category"]
    is_empty = row["empty_patch"]

    reason = None

    # Rule 1: remove by error category
    if cat in CATEGORIES_TO_REMOVE:
        reason = f"error:{cat}"

    # Rule 2: remove empty patches (unless agent legitimately ran)
    if is_empty and cat not in KEEP_EMPTY_PATCH_IF_CATEGORY:
        reason = f"empty_patch (error:{cat})"

    if reason:
        rerun_ids.add(iid)
        removal_reasons[iid] = reason

print(f"Instances to remove: {len(rerun_ids)} / {len(df)}")
print(f"Instances to keep:   {len(df) - len(rerun_ids)}")
print()

# Show removal breakdown
reason_counts = Counter(removal_reasons.values())
print("Removal reasons:")
for reason, cnt in reason_counts.most_common():
    print(f"  {reason}: {cnt}")

In [ ]:
# Preview: list all instances to be removed
print(f"Instances to remove ({len(rerun_ids)}):")
print()
for iid in sorted(rerun_ids):
    print(f"  {iid}  — {removal_reasons[iid]}")

## 6. Execute Removal

**⚠️ This step modifies files!** It will:
1. Back up `output.jsonl` and `output.swebench_eval.jsonl`
2. Remove the selected instances from both files
3. The next rollout will automatically pick up the missing instances

Run the cell below only when you're satisfied with the removal list above.

In [ ]:
def remove_instances_from_file(filepath: str, ids_to_remove: set[str]) -> tuple[int, int, str]:
    """Remove instances from a JSONL file. Returns (kept, removed, backup_path)."""
    if not os.path.exists(filepath):
        print(f"  File not found, skipping: {filepath}")
        return 0, 0, ""

    entries = load_output_file(filepath)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = filepath + f".backup_{timestamp}"
    shutil.copy2(filepath, backup)

    kept, removed = 0, 0
    with open(filepath, "w") as f:
        for raw, data in entries:
            if data is None:
                continue
            iid = str(data.get("instance_id", ""))
            if iid in ids_to_remove:
                removed += 1
                continue
            f.write(raw + "\n")
            kept += 1

    return kept, removed, backup


if not rerun_ids:
    print("Nothing to remove!")
else:
    print(f"Removing {len(rerun_ids)} instances...")
    print()

    # output.jsonl
    kept, removed, backup = remove_instances_from_file(OUTPUT_FILE, rerun_ids)
    print(f"output.jsonl: kept {kept}, removed {removed}")
    print(f"  Backup: {backup}")

    # .swebench_eval.jsonl
    if os.path.exists(EVAL_FILE):
        kept_e, removed_e, backup_e = remove_instances_from_file(EVAL_FILE, rerun_ids)
        print(f"swebench_eval.jsonl: kept {kept_e}, removed {removed_e}")
        print(f"  Backup: {backup_e}")
    else:
        print("No swebench_eval.jsonl found (skipped)")

    print()
    print("Done! Rerun the rollout script to process the removed instances.")

## 7. Validate After Removal

Reload the cleaned files and verify the removed instances are gone.

In [ ]:
# Reload and re-classify the cleaned output
if os.path.exists(OUTPUT_FILE):
    raw_entries_clean = load_output_file(OUTPUT_FILE)
    records_clean = []
    for raw, data in raw_entries_clean:
        if data is not None:
            records_clean.append(classify_instance(data))

    df_clean = pd.DataFrame(records_clean)
    print(f"Cleaned output.jsonl: {len(df_clean)} instances (was {len(df)})")
    print(f"Empty patches remaining: {df_clean['empty_patch'].sum()}")
    print()

    # Verify no removed IDs remain
    remaining_ids = set(df_clean["instance_id"])
    leaked = rerun_ids & remaining_ids
    if leaked:
        print(f"WARNING: {len(leaked)} removed IDs still present: {leaked}")
    else:
        print("All removed instances confirmed gone.")

    print()
    print("Remaining error breakdown:")
    cat_counts_clean = df_clean["error_category"].value_counts()
    for cat, cnt in cat_counts_clean.items():
        print(f"  {cat}: {cnt}")

    # Cross-tab for cleaned data
    ct_clean = pd.crosstab(df_clean["error_category"], df_clean["empty_patch"], margins=True, margins_name="Total")
    ct_clean.columns = ["has_patch", "empty_patch", "Total"]
    display(Markdown("### Cleaned: Error Category × Empty Patch"))
    display(ct_clean)
else:
    print("output.jsonl not found — cannot validate")